# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
# Feature Scaling
spaceship_clean = spaceship.dropna()

spaceship_clean.loc[:, 'Cabin'] = spaceship_clean['Cabin'].apply(lambda x: x[0])

spaceship_clean['Cabin'].value_counts()

spaceship_clean = spaceship_clean.drop(columns=['PassengerId', 'Name'])

spaceship_clean.dtypes

HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Transported        bool
dtype: object

In [4]:
# Featuring Selection

spaceship_clean = pd.get_dummies(spaceship_clean, columns=['Destination'])

spaceship_clean = pd.get_dummies(spaceship_clean, columns=['HomePlanet'])

spaceship_clean = pd.get_dummies(spaceship_clean, columns=['Cabin'])

- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [5]:

numerical_features = spaceship_clean.drop(columns=['Transported']).select_dtypes(include=['int64', 'float64'])

target = spaceship_clean['Transported']

X = numerical_features  # Features (all columns but the target)
y = target  # Target

X_train, X_test, y_train, y_test = train_test_split(numerical_features, target, test_size=0.2, random_state=42)

# test_size=0.2: Defines 20% of the data to be set aside for testing, ensuring that 80% is used for training.

# random_state=42: Sets a seed for reproducibility, meaning you’ll get the same split 
# every time you run the script (using any integer value can suffice, 42 is commonly used).

# Check the shape of the splits to confirm
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")



X_train shape: (5284, 6)
X_test shape: (1322, 6)
y_train shape: (5284,)
y_test shape: (1322,)


- Evaluate your model

In [ ]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

# BAGGING AND PASTING

# Initialize a base estimator, e.g., a Decision Tree
estimator = DecisionTreeClassifier(random_state=42)

# Bagging: use the default bootstrap=True
bagging_clf = BaggingClassifier(estimator=estimator, n_estimators=100, bootstrap=True, random_state=42)

# Pasting: set bootstrap=False
pasting_clf = BaggingClassifier(estimator=estimator, n_estimators=100, bootstrap=False, random_state=42)

# Fit the models
bagging_clf.fit(X_train, y_train)
pasting_clf.fit(X_train, y_train)

# predict and evaluate 
bagging_score = bagging_clf.score(X_test, y_test)
pasting_score = pasting_clf.score(X_test, y_test)

print(f'Bagging Test Score: {bagging_score}')
print(f'Pasting Test Score: {pasting_score}')

accuracies = {"Bagging": bagging_score, "Pasting": pasting_score}

Bagging Test Score: 0.7927382753403933
Pasting Test Score: 0.7503782148260212


In [7]:
# RANDOM FOREST

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Initialization
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)  # n_estimators = number of trees in the forest

# More trees can lead to a better model, but there is also more computational cost
# random_state: Ensures reproducibility by fixing the random seed.


rf_clf.fit(X_train, y_train)

# predictions on the test data
y_pred = rf_clf.predict(X_test)

# Evaluate 
random_forest_accuracy = accuracy_score(y_test, y_pred)
print(f'Random Forest Test Accuracy: {random_forest_accuracy:.2f}')

accuracies['Random Forest'] = random_forest_accuracy


Random Forest Test Accuracy: 0.79


In [8]:
# GRADIENT BOOSTING

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Initialization
gb_clf = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)

gb_clf.fit(X_train, y_train)

# Prediction
y_pred = gb_clf.predict(X_test)

#n_estimators: Number of boosting stages (trees) to be built. 
#learning_rate: Determines the contribution of each tree. A smaller value typically requires more trees.

# Evaluation
accuracy = accuracy_score(y_test, y_pred)
print(f'Gradient Boosting Test Accuracy: {accuracy:.2f}')

accuracies['Gradient Boosting Test Accuracy'] = accuracy

Gradient Boosting Test Accuracy: 0.80


In [9]:
# ADAPTATIVE BOOSTING

from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# Initialization
estimator = DecisionTreeClassifier(max_depth=1, random_state=42)

# AdaBoost Classifier
ada_clf = AdaBoostClassifier(estimator=estimator, n_estimators=100, learning_rate=0.1, random_state=42)


ada_clf.fit(X_train, y_train)

# Prediction
y_pred = ada_clf.predict(X_test)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)
print(f'AdaBoost Test Accuracy: {accuracy:.2f}')

accuracies['AdaBoost Test Accuracy'] = accuracy

/home/silvia/anaconda3/lib/python3.12/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


AdaBoost Test Accuracy: 0.79


In [10]:
accuracies = {k: f"{v*100:.2f}%" for k, v in accuracies.items()}

accuracies

{'Bagging': '79.27%',
 'Pasting': '75.04%',
 'Random Forest': '79.35%',
 'Gradient Boosting Test Accuracy': '79.58%',
 'AdaBoost Test Accuracy': '79.35%'}

### We can see that the best accuracy we get it from the Gradient Boosting.

**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [11]:
param_grid = {
    'n_estimators': [100, 200, 500],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 4, 8],
    'subsample': [0.8, 1.0],
    'min_samples_split': [2, 5, 10]
}

gb_clf = GradientBoostingClassifier(random_state=42)

- Run Grid Search

In [12]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

grid_search = GridSearchCV(estimator=gb_clf, param_grid=param_grid, cv=3, scoring='accuracy', verbose=2, n_jobs=-1)

## cv=3: validación cruzada de 3 pliegues.
## scoring='accuracy': Optimiza la precisión. 
## n_jobs=-1: Usa todos los procesadores disponibles para acelerar el proceso.

grid_search.fit(X_train, y_train)

# mejores parámetros
best_params = grid_search.best_params_
print("Mejores parámetros encontrados:", best_params)

# Reentrenar 
best_model = grid_search.best_estimator_

Fitting 3 folds for each of 162 candidates, totalling 486 fits
[CV] END learning_rate=0.01, max_depth=3, min_samples_split=2, n_estimators=100, subsample=0.8; total time=   0.7s
[CV] END learning_rate=0.01, max_depth=3, min_samples_split=2, n_estimators=100, subsample=0.8; total time=   0.6s
[CV] END learning_rate=0.01, max_depth=3, min_samples_split=2, n_estimators=100, subsample=1.0; total time=   0.7s
[CV] END learning_rate=0.01, max_depth=3, min_samples_split=2, n_estimators=100, subsample=0.8; total time=   0.8s
[CV] END learning_rate=0.01, max_depth=3, min_samples_split=2, n_estimators=100, subsample=1.0; total time=   0.7s
[CV] END learning_rate=0.01, max_depth=3, min_samples_split=2, n_estimators=100, subsample=1.0; total time=   0.6s
[CV] END learning_rate=0.01, max_depth=3, min_samples_split=2, n_estimators=200, subsample=0.8; total time=   1.1s
[CV] END learning_rate=0.01, max_depth=3, min_samples_split=2, n_estimators=200, subsample=0.8; total time=   1.4s
[CV] END learning

Fitting 3 folds for each of 162 candidates, totalling 486 fits

Mejores parámetros encontrados: {'learning_rate': 0.01, 'max_depth': 3, 'min_samples_split': 5, 'n_estimators': 500, 'subsample': 1.0}

- Evaluate your model

In [13]:
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy del modelo optimizado: {accuracy:.2f}")

Accuracy del modelo optimizado: 0.80


#### We can see that it has been an improvement in the accuracy